# TT truncation: rank / 再構成誤差 / 保存量のtrade-off

TT-SVDでrank上限を変えたときの、**実際のTT-rank列・保存量・relative Frobenius error** を比較する。

## このNotebookで何を確認するか

01では各段階の数値rankをすべて残すTT-SVDを作り、元テンソルをほぼ完全に再構成した。ここでは各bond dimensionに `max_rank` という上限を設け、**あえてrankを削って情報を捨てる**。

rank上限を小さくするとTT coreの保存量は減る一方で、再構成誤差は増える。このtrade-offを測り、TTを単なる分解ではなく**圧縮方法として使うと何が起こるか**を確認する。

- `## 2` で rank 制限付きTT-SVDを作る
- `## 4` で `max_rank` を変え、圧縮量と誤差を比較する

## ゴール
- `max_rank` を変えてTT-SVDする
- 実際の `bond_ranks` を記録する
- TTパラメータ数と圧縮率を求める
- relative Frobenius errorを比較する


## 1. 実験対象

$X \in \mathbb{R}^{4 \times 4 \times 4 \times 4}$ を使う。

**このセルの目的:** 圧縮前のdense tensorを1つ用意する。以降は同じ $X$ に対してrank上限だけを変え、`max_rank` の違いだけで保存量と再構成誤差がどう変わるか比較する。

ここで作る $X$ の要素数は、後でTT coreの総要素数と比較するときの基準にもなる。


In [19]:
import torch
import math
import pandas as pd
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)
torch.manual_seed(2)

# TODO
X = torch.randn(4, 4, 4,4)


## 2. truncation付きTT-SVD

**このセルの目的:** 01で作った打ち切りなしTT-SVDを、**bond rankに上限を付けられる圧縮用TT-SVD**へ変える。

01では各段階の数値rankをすべて残したが、ここでは必要なrankが大きくても `max_rank` までしか残さない。これによってTT coreは小さくなる一方、捨てた特異値の分だけ元の $X$ を完全には表せなくなる。

この関数の出力は近似TT core列であり、後のセルで再構成して誤差と保存量を測る。

```python
tt_svd(X, max_rank)
```

各段階で

$$
\widetilde r_k = \min\left(\text{max_rank},\ \text{その段階の数値rank}\right)
$$

とする。


In [20]:
from nn_compression.tensor.validation import (
    validate_tensor_shape,
)
from nn_compression.compression import (
    truncated_svd,
)
from nn_compression.metrics import relative_frobenius_error

In [21]:
def tt_svd(X: torch.Tensor, max_rank: int) -> list[torch.Tensor]:
    validate_tensor_shape(X)

    shape = X.shape
    d = X.ndim
    cores: list[torch.Tensor] = []

    remainder = X
    r_left = 1

    for mode in range(d - 1):
        n_mode = shape[mode]
        mat = remainder.reshape(r_left * n_mode, -1)

        numerical_rank = int(torch.linalg.matrix_rank(mat).item())
        r = min(max_rank, numerical_rank)
        U, S, Vh = truncated_svd(mat, r)
        cores.append(U.reshape(r_left, n_mode, r))

        remainder = (torch.diag(S) @ Vh).reshape(r, *shape[mode + 1 :])
        r_left = r

    cores.append(remainder.reshape(r_left, shape[-1], 1))
    return cores


## 3. 補助関数

**このセルの目的:** `tt_svd` が作ったTT core列を、**圧縮結果として評価するための道具**を用意する。

- `tt_reconstruct`: TT core列から $\hat X$ を作る。元の $X$ と比較するために必要。
- `relative_frobenius_error`: $X$ と $\hat X$ の差を1つの数値にして、rankを削ったことによる情報損失を測る。
- `tt_num_parameters`: TT coreに何要素保存しているか数え、dense表現よりどれだけ保存量が変わったか測る。

つまりこのセルでは、**近似を作る処理ではなく、作った近似を測る処理**を準備する。

$$
\frac{\|X - \hat X\|_F}{\|X\|_F},
\qquad
P_{\mathrm{TT}} = \sum_k r_{k-1} n_k r_k
$$


In [22]:
def tt_reconstruct(cores: list[torch.Tensor]) -> torch.Tensor:
    """TT core 列を左から bond 縮約し、dense テンソルを再構成する。"""
    result = cores[0]
    for core in cores[1:]:
        # 左 core の右 bond（最終軸）と右 core の左 bond（先頭軸）を縮約
        result = torch.tensordot(result, core, dims=([-1], [0]))
    return result.squeeze(0).squeeze(-1)


def tt_num_parameters(cores: list[torch.Tensor]) -> int:
    return sum(core.numel() for core in cores)




## 4. rank sweep

**このセルの目的:** 同じ $X$ に対して `max_rank` だけを変え、**圧縮の強さと結果の違いを表にまとめる**。

各 `max_rank` についてTT-SVDし、再構成した $\hat X$、実際にできたbond rank列、保存量、圧縮率、再構成誤差を記録する。ここで初めて「rankをどこまで削ると、どれだけ小さくなり、どれだけ精度を失うか」を横並びで比較できる。

`max_ranks = [1, 2, 4, 8]` を試し、次を記録する。

- `max_rank`
- `bond_ranks`
- `tt_params`
- `compression_ratio`
- `relative_error`

`max_rank` はあくまで上限なので、元々の数値rankがそれより小さいcutでは、実際の `bond_ranks` は `max_rank` より小さくなる。


In [23]:
max_ranks = [1, 2, 4, 8]
results = []

for max_rank in max_ranks:
    cores = tt_svd(X, max_rank)
    X_hat = tt_reconstruct(cores)
    bond_ranks = [core.shape[2] for core in cores[:-1]]
    tt_params = tt_num_parameters(cores)
    rel_error = relative_frobenius_error(X, X_hat)
    compression_ratio = tt_params / X.numel()
    results.append({
        "max_rank": max_rank,
        "bond_ranks": bond_ranks,
        "tt_params": tt_params,
        "compression_ratio": compression_ratio,
        "relative_error": rel_error.item(),
    })

df = pd.DataFrame(results)
df


,max_rank,bond_ranks,tt_params,compression_ratio,relative_error
0,1,"[1, 1, 1]",16,0.0625,0.942147
1,2,"[2, 2, 2]",48,0.1875,0.850899
2,4,"[4, 4, 4]",160,0.6250,0.614882
3,8,"[4, 8, 4]",288,1.1250,0.294857


## 5. 可視化

**この2つのセルの目的:** `## 4` の表をグラフにして、rank上限を変えたときのtrade-offを見やすくする。

1つ目は `max_rank` と `relative_error` を描き、**rankを多く残すと近似誤差がどう変わるか**を見る。

2つ目は `max_rank` と `compression_ratio` を描き、**rankを多く残すと保存量の削減効果がどう変わるか**を見る。

この2枚を対応させて見ることで、「誤差を小さくしたい」と「強く圧縮したい」が両立しにくいことを確認する。


In [24]:
fig, ax = plt.subplots()
ax.plot(df["max_rank"], df["relative_error"], marker="o")
ax.set_xlabel("max_rank")
ax.set_ylabel("relative_error")
ax.set_title("max_rank vs relative_error")
plt.show()


In [25]:
fig, ax = plt.subplots()
ax.plot(df["max_rank"], df["compression_ratio"], marker="o")
ax.set_xlabel("max_rank")
ax.set_ylabel("compression_ratio (tt_params / dense)")
ax.set_title("max_rank vs compression_ratio")
plt.show()


## 6. 考察

**このセルの目的:** 計算結果を出して終わりにせず、TT圧縮でrankが果たしている役割を言葉で整理する。

- rank上限を下げたとき、なぜ誤差が増えるのか
- `max_rank` と実際の `bond_ranks` がなぜ一致しない場合があるのか
- rank上限を変えると保存量・圧縮率がどう変わるのか
- 小テンソルでは、TT coreの総要素数がdenseより多くなり、TTが必ずしも圧縮にならないのはなぜか

を結果と対応させて整理する。
